## Notebook 2: SQL Analysis

1. What does the average user's activity look like?
2. Which users are most/least active?
3. Is there a relationship between steps and sleep?
4. Do more active users have lower resting heart rates?
5. What days of the week are people most active?


## 1. Setup

connect to PostgreSQL database

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# Connect to PostgreSQL
engine = create_engine("postgresql://luismillan@localhost/fitbit_health")

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM daily_metrics"))
    print("Total rows:", result.fetchone()[0])

Total rows: 1388


## 2. User Activity Overview

### 2.1 Average activty metrics per user

In [2]:
query = """
SELECT 
    id,
    ROUND(AVG(total_steps)) AS avg_daily_steps,
    ROUND(AVG(calories)) AS avg_daily_calories,
    ROUND(AVG(very_active_minutes)) AS avg_very_active_mins,
    ROUND(AVG(sedentary_minutes)) AS avg_sedentary_mins,
    COUNT(*) AS days_tracked
FROM daily_metrics
GROUP BY id
ORDER BY avg_daily_steps DESC
"""

user_activity = pd.read_sql(query, engine)
print(user_activity.to_string())

            id  avg_daily_steps  avg_daily_calories  avg_very_active_mins  avg_sedentary_mins  days_tracked
0   8877689391          16424.0              3429.0                  66.0              1094.0            43
1   8053475328          14785.0              2932.0                  85.0              1139.0            42
2   1503960366          12179.0              1846.0                  38.0               821.0            49
3   7007744171          11619.0              2570.0                  35.0              1038.0            38
4   2022484408          11595.0              2500.0                  37.0              1098.0            43
5   6962181067          10680.0              2015.0                  27.0               645.0            45
6   3977333714          10322.0              1481.0                  17.0               708.0            42
7   2347167796           9647.0              2033.0                  13.0               686.0            33
8   4388161847           859

## 3. Activity vs SLeep Correlation

Do users who take more steps sleep longer and better?

In [3]:
query = """
SELECT 
    total_steps,
    total_sleep_minutes,
    restless_minutes,
    awake_minutes,
    ROUND((asleep_minutes / NULLIF(total_sleep_minutes, 0) * 100)::numeric, 1) AS sleep_efficiency_pct
FROM daily_metrics
WHERE total_sleep_minutes IS NOT NULL
  AND total_sleep_minutes > 0
ORDER BY total_steps DESC
"""

activity_sleep = pd.read_sql(query, engine)
print(activity_sleep.shape)
activity_sleep.head(10)

(645, 5)


,total_steps,total_sleep_minutes,restless_minutes,awake_minutes,sleep_efficiency_pct
0,22770,496.0,24.0,0.0,95.2
1,22359,337.0,1.0,5.0,98.2
2,22244,66.0,4.0,0.0,93.9
3,20031,472.0,4.0,0.0,99.2
4,19769,75.0,0.0,1.0,98.7
5,19658,466.0,12.0,0.0,97.4
6,19542,537.0,36.0,5.0,92.4
7,18952,399.0,32.0,4.0,91.0
8,18213,142.0,15.0,3.0,87.3
9,17609,472.0,31.0,4.0,92.6


## 4. Activity Buckets vs Sleep Quality
Categorizing users by average daily steps and compare sleep metrics 

In [4]:
query = """
SELECT
    CASE
        WHEN total_steps >= 10000 THEN 'Very Active (10k+)'
        WHEN total_steps >= 7500  THEN 'Active (7.5k-10k)'
        WHEN total_steps >= 5000  THEN 'Moderate (5k-7.5k)'
        ELSE 'Sedentary (<5k)'
    END AS activity_level,
    COUNT(*) AS days,
    ROUND(AVG(total_sleep_minutes)::numeric, 1) AS avg_sleep_mins,
    ROUND(AVG(asleep_minutes / NULLIF(total_sleep_minutes, 0) * 100)::numeric, 1) AS avg_sleep_efficiency,
    ROUND(AVG(restless_minutes)::numeric, 1) AS avg_restless_mins
FROM daily_metrics
WHERE total_sleep_minutes IS NOT NULL
  AND total_sleep_minutes > 0
GROUP BY activity_level
ORDER BY avg_sleep_mins DESC
"""

buckets = pd.read_sql(query, engine)
buckets

,activity_level,days,avg_sleep_mins,avg_sleep_efficiency,avg_restless_mins
0,Sedentary (<5k),169,478.4,91.4,32.8
1,Moderate (5k-7.5k),117,430.7,92.9,25.3
2,Very Active (10k+),247,422.4,90.3,37.2
3,Active (7.5k-10k),112,413.5,91.2,27.8


## 5. Day of Week Analysis
Which days are users most and least active?

In [5]:
query = """
SELECT
    TO_CHAR(activity_date, 'Day') AS day_of_week,
    EXTRACT(DOW FROM activity_date) AS day_num,
    ROUND(AVG(total_steps)::numeric, 0) AS avg_steps,
    ROUND(AVG(calories)::numeric, 0) AS avg_calories,
    ROUND(AVG(very_active_minutes)::numeric, 1) AS avg_very_active_mins
FROM daily_metrics
GROUP BY day_of_week, day_num
ORDER BY day_num
"""

dow = pd.read_sql(query, engine)
dow

,day_of_week,day_num,avg_steps,avg_calories,avg_very_active_mins
0,Sunday,0.0,6641.0,2239.0,18.3
1,Monday,1.0,7541.0,2298.0,21.6
2,Tuesday,2.0,7212.0,2196.0,20.1
3,Wednesday,3.0,7548.0,2321.0,19.9
4,Thursday,4.0,7344.0,2247.0,19.1
5,Friday,5.0,7188.0,2325.0,18.6
6,Saturday,6.0,7831.0,2349.0,21.0


## 6. Heart Rate vs Activity Level
Do more active users have lower average heart rates?

In [6]:
query = """
SELECT
    CASE
        WHEN total_steps >= 10000 THEN 'Very Active (10k+)'
        WHEN total_steps >= 7500  THEN 'Active (7.5k-10k)'
        WHEN total_steps >= 5000  THEN 'Moderate (5k-7.5k)'
        ELSE 'Sedentary (<5k)'
    END AS activity_level,
    COUNT(*) AS days,
    ROUND(AVG(avg_heart_rate)::numeric, 1) AS avg_resting_hr,
    ROUND(AVG(max_heart_rate)::numeric, 1) AS avg_max_hr,
    ROUND(AVG(total_steps)::numeric, 0) AS avg_steps
FROM daily_metrics
WHERE avg_heart_rate IS NOT NULL
GROUP BY activity_level
ORDER BY avg_steps DESC
"""

hr_activity = pd.read_sql(query, engine)
hr_activity

,activity_level,days,avg_resting_hr,avg_max_hr,avg_steps
0,Very Active (10k+),234,80.3,148.1,13415.0
1,Active (7.5k-10k),65,77.9,139.2,8846.0
2,Moderate (5k-7.5k),75,78.7,131.7,6209.0
3,Sedentary (<5k),104,77.1,122.8,2604.0
